# MPC spatial scene guide (waypoints → problem → solve)

Pedagogical walkthrough for a rate-input dynamic bicycle:

1. **Workspace** — waypoints, `ReferenceTrack`, sphere obstacles, `Scene` (+ optional terrain).
2. **Body** — `bind(sys, geometry)` turns state `x` into workspace probes.
3. **State fields** — `clearance_field`, `distance_field`, `corridor_field`, `cost_field` expose `value(x)`.
4. **Exports** — `as_constraint()` for hard tubes / free space; `as_cost(shaping=...)` for soft penalties.
5. **Assembly** — `PlanningProblem` with composed `X` and `cost`.
6. **Transcription → NLP** — manual steps: `transcribe` → `MathematicalProgram` → `Optimizer` → `Trajectory`.
7. **Planner wrapper** — `TrajectoryOptimizationPlanner.compute_solution()` orchestrates the same chain.
8. **MPC** — closed-loop receding horizon (last section).

Companion script (spatial steps only, no solve): `examples/scripts/mpc/demo_mpc_spatial_scene_guide.py`

Run from repo root with `pip install -e ".[jax]"`.

### Two domains (read this once)

Spatial planning uses **two coordinates**:

| Domain | Symbol | Objects | Example query |
| --- | --- | --- | --- |
| **Workspace** | `p ∈ ℝ²` | waypoints, obstacles, terrain | `scene.clearance(p)`, `track.distance(p)` |
| **State** | `x ∈ ℝⁿ` | full vehicle pose + rates | `field.value(x)` after `bind(sys, body)` |

The bridge is forward kinematics: `bind` places body probes in the workspace from `x`. Costs and constraints are exported on **state**; heatmaps rasterize **workspace** points with `point_probe`.

In [ ]:
# %matplotlib inline

# Colab setup (uncomment):
# !git clone https://github.com/alx87grd/minilink.git
# %cd minilink
# !pip install -q -e ".[jax]"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.backends import configure_jax
from minilink.core.costs import QuadraticCost
from minilink.core.geometry import Sphere
from minilink.core.sets import BoxSet
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.vehicles.dynamic_bicycle import JaxDynamicBicycleRateInputs
from minilink.graphical.animation.primitives import HorizonPolyline, TrajectoryPolyline
from minilink.graphical.catalog import SceneHistory
from minilink.optimization.evaluators.compiler import compile_program_evaluator
from minilink.optimization.optimizer import Optimizer
from minilink.planning.initial_guess import default_initial_trajectory
from minilink.planning.problems import PlanningProblem
from minilink.planning.spatial.collision import bind, car_outline, point_probe
from minilink.planning.spatial.grid import pad_bounds, sample_field_costs
from minilink.planning.spatial.paths import from_waypoints
from minilink.planning.spatial.plotting import plot_cost_field_exports
from minilink.planning.spatial.scene import Scene
from minilink.planning.spatial.shaping import (
    inverse_barrier,
    occupancy,
    quadratic_excess,
    quadratic_hinge,
)
from minilink.planning.spatial.track import ReferenceTrack
from minilink.planning.spatial.workspace_fields import GaussianField
from minilink.planning.trajectory_optimization.direct_collocation import (
    DirectCollocationOptions,
    DirectCollocationTranscription,
)
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationOptions,
    TrajectoryOptimizationPlanner,
)

## Scenario parameters

A compact CCW loop, three keepout spheres, and a soft Gaussian terrain patch.

In [ ]:
WAYPOINTS = np.array(
    [
        [-8.0, -5.0],
        [8.0, -5.0],
        [12.0, 0.0],
        [8.0, 5.0],
        [-8.0, 5.0],
        [-12.0, 0.0],
        [-8.0, -5.0],
    ]
)
CORRIDOR_HALF_WIDTH = 2.0
OBSTACLE_RADIUS = 0.35
OBSTACLE_MARGIN = 0.15
OBSTACLE_CENTERS = ((-4.0, -4.0), (4.0, 0.0), (0.0, 4.0))
TERRAIN_CENTER = (2.0, 2.0)

PATH_COST_WEIGHT = 30.0
CORRIDOR_COST_WEIGHT = 20.0
OBSTACLE_REPULSION_WEIGHT = 25.0
TERRAIN_COST_WEIGHT = 5.0
U_TARGET = 12.0

PLOT_MARGIN = 2.0
COST_FIELD_MARGIN = 4.0
PLOT_BOUNDS = (
    (WAYPOINTS[:, 0].min() - PLOT_MARGIN, WAYPOINTS[:, 0].max() + PLOT_MARGIN),
    (WAYPOINTS[:, 1].min() - PLOT_MARGIN, WAYPOINTS[:, 1].max() + PLOT_MARGIN),
)

### Map the scenario

Plot the geometry before attaching a robot — workspace-only objects.

In [ ]:
_preview_track = ReferenceTrack(from_waypoints(WAYPOINTS), half_width=CORRIDOR_HALF_WIDTH)
_preview_scene = Scene(
    obstacles=tuple(Sphere(c, OBSTACLE_RADIUS + OBSTACLE_MARGIN) for c in OBSTACLE_CENTERS),
    workspace_fields=(GaussianField(TERRAIN_CENTER, amplitude=1.0, sigma=2.5),),
)
_, ax = _preview_scene.plot(show=False, bounds=PLOT_BOUNDS, show_density=True, title="")
_preview_track.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, title="")
ax.set_title("Workspace: track tube + keepouts + terrain")
plt.show()

## Step 1 — Planner plant

`JaxDynamicBicycleRateInputs`: wheel speed and steer angle are **states**; their rates are **inputs**.

In [ ]:
configure_jax(enable_x64=True)

W_REAR_MAX = 90.0
DELTA_MAX = 0.55
W_REAR_DOT_MAX = 80.0
DELTA_DOT_MAX = 2.0

sys_mpc = JaxDynamicBicycleRateInputs()
sys_mpc.state.lower_bound[6] = 0.0
sys_mpc.state.upper_bound[6] = W_REAR_MAX
sys_mpc.state.lower_bound[7] = -DELTA_MAX
sys_mpc.state.upper_bound[7] = DELTA_MAX
sys_mpc.inputs["w_rear_dot"].lower_bound[0] = -W_REAR_DOT_MAX
sys_mpc.inputs["w_rear_dot"].upper_bound[0] = W_REAR_DOT_MAX
sys_mpc.inputs["delta_dot"].lower_bound[0] = -DELTA_DOT_MAX
sys_mpc.inputs["delta_dot"].upper_bound[0] = DELTA_DOT_MAX
print(f"n={sys_mpc.n}, m={sys_mpc.m}")

## Step 2 — Waypoints → `ReferenceTrack`

`from_waypoints` builds a workspace polyline. `ReferenceTrack` adds a constant half-width corridor.

Workspace queries (no robot yet):
- `track.distance(p)` — meters to centerline
- `track.corridor_margin(p)` — `half_width - distance` (positive inside the tube)

In [ ]:
track = ReferenceTrack(from_waypoints(WAYPOINTS), half_width=CORRIDOR_HALF_WIDTH)
sample_p = np.array([0.0, -4.0])
print(f"path length = {track.path.total_length:.2f} m")
print(f"distance({sample_p}) = {track.distance(sample_p):.3f}")
print(f"corridor_margin({sample_p}) = {track.corridor_margin(sample_p):.3f}")

## Step 3 — `Scene`: obstacles and terrain

Hard shapes live in `scene.obstacles`. Soft penalties live in `scene.workspace_fields`.

Workspace queries:
- `scene.clearance(p)` — signed distance to nearest obstacle (positive in free space)
- `scene.cost_density(p)` — sum of terrain penalty densities

Factories (need a collision **body** next):
- `scene.clearance_field(body)` → `ClearanceField`
- `scene.cost_field(body)` → `CostDensityField`

In [ ]:
keepout_radius = OBSTACLE_RADIUS + OBSTACLE_MARGIN
scene = Scene(
    obstacles=tuple(Sphere(c, keepout_radius) for c in OBSTACLE_CENTERS),
    workspace_fields=(GaussianField(TERRAIN_CENTER, amplitude=1.0, sigma=2.5),),
)
print(f"clearance({sample_p}) = {scene.clearance(sample_p):.3f}")
print(f"cost_density({sample_p}) = {scene.cost_density(sample_p):.3f}")

## Step 4 — Collision body: `bind(sys, geometry)`

Geometry is **frameless** (`point_probe`, `disc`, `car_outline`). `bind` attaches it to the planner plant so forward kinematics place probes in the workspace — the same transform used for rendering.

- **MPC planner** — `car_outline` (oriented footprint)
- **Heatmaps** — `point_probe` (workspace position only, heading-independent)

In [ ]:
body = bind(sys_mpc, car_outline(length=2.2, width=0.2, margin=0.05))
probe = bind(sys_mpc, point_probe())

r_r = sys_mpc.params["r_r"]
w_rear_ref = U_TARGET / r_r
x_cruise = np.array([0.0, -4.0, 0.0, U_TARGET, 0.0, 0.0, w_rear_ref, 0.0])
ubar = np.zeros(sys_mpc.m)

### Probe vs body at the same pose

`car_outline` subtracts footprint extent; `point_probe` is a single world point. Heatmaps use `point_probe` so tuning plots do not depend on heading.

In [ ]:
_clear_body = scene.clearance_field(body).value(x_cruise)
_clear_probe = scene.clearance_field(probe).value(x_cruise)
print(f"clearance @ x_cruise  car_outline = {_clear_body:.3f} m")
print(f"clearance @ x_cruise  point_probe = {_clear_probe:.3f} m")
print(f"delta (body more conservative) = {_clear_probe - _clear_body:.3f} m")

## Step 5 — State fields: `value(x)`

Each field fuses workspace queries over all body probes (min clearance / distance, max terrain density).

Track factories mirror the scene pattern:
- `track.distance_field(body)` → `PathDistanceField`
- `track.corridor_field(body)` → `CorridorMarginField`

In [ ]:
clearance_field = scene.clearance_field(body)
corridor_field = track.corridor_field(body)
distance_field = track.distance_field(body)
terrain_field = scene.cost_field(body)

for name, field in [
    ("clearance", clearance_field),
    ("corridor", corridor_field),
    ("path distance", distance_field),
    ("terrain", terrain_field),
]:
    print(f"{name:14s} value(x_cruise) = {float(field.value(x_cruise)):.4f}")

## Step 6 — Hard set: `field.as_constraint()`

`FieldSet` is feasible where `lower <= value(x) <= upper`. Compose with `&`:

```
X = state_bounds & obstacle_free & corridor_tube
```

Direct collocation enforces `X.contains(x_k)` at each node. MPC demos often skip hard `X` and rely on soft costs only; both patterns are shown here.

### Hard vs soft — when to use which

| Export | Feasibility | Typical use |
| --- | --- | --- |
| `as_constraint(lower=0)` | **Must** satisfy at every collocation node | corridor tube, collision-free band |
| `as_cost(shaping=...)` | Penalty only — solver may trade off | path centering, obstacle repulsion, comfort |

This notebook uses **both**: hard `X` for the tube and obstacles, soft costs for tracking and barriers. Many MPC demos drop hard spatial `X` and rely on weighted soft costs only — easier to tune, no guaranteed feasibility.

In [ ]:
X = (
    BoxSet.from_system_state(sys_mpc)
    & clearance_field.as_constraint(lower=0.0)
    & corridor_field.as_constraint(lower=0.0)
)
print("X assembled: bounds ∩ obstacle clearance ≥ 0 ∩ corridor margin ≥ 0")

## Step 7 — Soft cost: `field.as_cost(shaping=...)`

Shaping maps the raw field value to a penalty **before** weighting:

| Shaping | Use with | Effect |
| --- | --- | --- |
| `quadratic_excess` | path distance | quadratic once farther than threshold from centerline |
| `quadratic_hinge` | corridor margin | quadratic once outside the tube |
| `inverse_barrier` | clearance | blows up as clearance → 0 |
| `occupancy` | terrain density | bounded sigmoid in [0, 1] |

Compose with `+` on `CostFunction` terms.

In [ ]:
stability_cost = QuadraticCost.from_system(
    sys_mpc,
    Q=np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0]),
    R=np.diag([1.0, 22.0]),
    S=np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0]),
    xbar=x_cruise,
    ubar=ubar,
)
path_cost = distance_field.as_cost(
    weight=PATH_COST_WEIGHT, shaping=quadratic_excess(threshold=0.1)
)
corridor_cost = corridor_field.as_cost(
    weight=CORRIDOR_COST_WEIGHT, shaping=quadratic_hinge(threshold=0.0)
)
obstacle_cost = clearance_field.as_cost(
    weight=OBSTACLE_REPULSION_WEIGHT, shaping=inverse_barrier(epsilon=0.08)
)
terrain_cost = terrain_field.as_cost(
    weight=TERRAIN_COST_WEIGHT, shaping=occupancy(scale=1.0)
)
cost = stability_cost + path_cost + corridor_cost + obstacle_cost + terrain_cost

In [ ]:
_v = np.linspace(-0.5, 3.0, 200)
fig, ax = plt.subplots(figsize=(7.0, 3.5))
ax.plot(_v, [float(quadratic_excess(0.1)(vi)) for vi in _v], label="quadratic_excess (path)")
ax.plot(_v, [float(quadratic_hinge(0.0)(vi)) for vi in _v], label="quadratic_hinge (corridor)")
ax.plot(_v, [float(inverse_barrier(0.08)(vi)) for vi in _v], label="inverse_barrier (obstacle)")
ax.plot(_v, [float(occupancy(1.0)(vi)) for vi in _v], label="occupancy (terrain)")
ax.set_xlabel("raw field value v")
ax.set_ylabel("shaping(v)")
ax.set_title("Shaping functions (before weighting)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 — `PlanningProblem`

The **continuous** object passed to transcription. Receding-horizon MPC rebuilds it each tick with the measured `x_start`.

In [ ]:
problem = PlanningProblem(sys=sys_mpc, x_start=x_cruise, X=X, cost=cost)
print(problem)

## Step 9 — Transcription: continuous → finite NLP

Architecture contract (see `DESIGN.md`):

```
PlanningProblem  →  Transcription.transcribe()  →  MathematicalProgram
                                                      ↓
pack_initial_guess → z0  →  Optimizer.solve()  →  OptimizationResult
                                                      ↓
                              reconstruct_result()  →  Trajectory
```

**Direct collocation** picks a fixed time grid `t[0], …, t[N-1]` and packs all sampled states and inputs into one decision vector:

```
z = [x[0,:], …, x[n-1,:], u[0,:], …, u[m-1,:]]   # shape (n_z,)
```

The `MathematicalProgram` is the textbook NLP

```
minimize   J(z)
subject to h(z) = 0        # trapezoidal dynamics defects + boundary equalities
           g(z) ≥ 0        # path/corridor/obstacle margins from problem.X
           lower ≤ z ≤ upper
```

`MathematicalProgram` stores only these callables — no solver state. `Optimizer` compiles gradients/Jacobians and calls SciPy (or IPOPT).

In [ ]:
MPC_HORIZON = 2.0
MPC_STEPS = 20
MPC_MAXITER = 120
MPC_FTOL = 1e-1
COMPILE_BACKEND = "jax"

transcription = DirectCollocationTranscription(
    DirectCollocationOptions(tf=MPC_HORIZON, n_steps=MPC_STEPS)
)
trajopt_options = TrajectoryOptimizationOptions(
    compile_backend=COMPILE_BACKEND,
    optimizer_method="scipy_slsqp",
    solve_disp=False,
    record_solve_time=True,
    optimizer_options={"maxiter": MPC_MAXITER, "ftol": MPC_FTOL},
)
n, m, n_steps = sys_mpc.n, sys_mpc.m, MPC_STEPS
print(f"horizon={MPC_HORIZON}s, N={n_steps} nodes")
print(f"decision dim n_z = (n + m) * N = ({n} + {m}) * {n_steps} = {(n + m) * n_steps}")

### Step 9a — `transcribe(problem)` → `MathematicalProgram`

The transcription reads `problem.cost`, `problem.X`, `problem.X0`/`Xf`, and `problem.sys.f`, then builds `J`, `h`, `g`, and box bounds on `z`.

In [ ]:
program = transcription.transcribe(problem, compile_backend=COMPILE_BACKEND)
evaluator = compile_program_evaluator(program, sample_z=np.zeros(program.n_z))
print("MathematicalProgram")
print(f"  n_z   = {program.n_z}")
print(f"  n_h   = {evaluator.n_h}  (equalities: dynamics defects + boundaries)")
print(f"  n_g   = {evaluator.n_g}  (inequalities: path/corridor/obstacle margins)")
print(f"  backend = {program.backend!r}")

### Step 9b — Initial guess on the collocation grid

`initial_guess_time_grid` returns `t = linspace(0, tf, N)`. `default_initial_trajectory` interpolates from `x_start` toward `x_goal` (or holds `x_start` when no goal). `pack_initial_guess` flattens `(x, u)` into `z0`.

In [ ]:
t_grid = transcription.initial_guess_time_grid(problem)
guess = default_initial_trajectory(problem, t_grid)
z0 = transcription.pack_initial_guess(problem, guess)

evaluator = compile_program_evaluator(program, sample_z=z0)
J0 = float(evaluator.objective(z0))
max_eq, min_ineq, max_bound = evaluator.constraint_violations(z0)
print(f"t_grid: {t_grid[0]:.2f} … {t_grid[-1]:.2f} s  ({t_grid.size} nodes)")
print(f"z0 shape = {z0.shape}")
print(f"J(z0) = {J0:.4f}")
print(f"violations @ z0: max_eq={max_eq:.2e}, min_ineq={min_ineq:.2e}, max_bound={max_bound:.2e}")

**Reading constraint violations**

- `max_eq` — largest |equality residual|; should be ≈ 0 when dynamics and pinned `x_start` are satisfied.
- `min_ineq` — smallest inequality margin `g(z)`; should be ≥ 0 when corridor/obstacle constraints are feasible.
- `max_bound` — box-bound slack violation on `z`.

Decode `z0` to confirm the first collocation state matches `x_start`:

In [ ]:
x0_unpack, u0_unpack = transcription.unpack(z0, problem)
print("x[:, 0] from z0:", np.round(x0_unpack[:, 0], 3))
print("x_start:       ", np.round(problem.x_start, 3))
print("match:", np.allclose(x0_unpack[:, 0], problem.x_start, atol=1e-9))

### Step 9c — `Optimizer(program, z0).solve()` → `OptimizationResult`

`Optimizer` compiles `J`, `h`, `g` (and derivatives) into a backend evaluator, then dispatches to the requested solver method.

In [ ]:
optimizer = Optimizer(
    program,
    z0=z0,
    method=trajopt_options.optimizer_method,
    use_hessian=trajopt_options.use_hessian,
    options=dict(trajopt_options.optimizer_options),
)
manual_result = optimizer.solve(record_solve_time=True)

max_eq, min_ineq, max_bound = optimizer.program_evaluator.constraint_violations(
    manual_result.z
)
print(f"success = {manual_result.success}")
print(f"message = {manual_result.message}")
print(f"J* = {manual_result.cost:.4f}")
print(f"solve_time = {manual_result.solve_time_s:.3f} s")
print(f"violations @ z*: max_eq={max_eq:.2e}, min_ineq={min_ineq:.2e}, max_bound={max_bound:.2e}")

### Step 9d — `reconstruct_result` → `Trajectory`

Unpack `z*` back into `(x, u)` matrices on the collocation grid and attach `dx` (and per-node cost if available).

In [ ]:
manual_traj = transcription.reconstruct_result(
    manual_result,
    problem=problem,
    compile_backend=COMPILE_BACKEND,
)
print(f"trajectory: {manual_traj.n_samples} samples, tf={manual_traj.t[-1]:.2f} s")
print(f"x(0)  xy = ({manual_traj.x[0,0]:.2f}, {manual_traj.x[1,0]:.2f})")
print(f"x(tf) xy = ({manual_traj.x[0,-1]:.2f}, {manual_traj.x[1,-1]:.2f})")

_, ax = scene.plot(show=False, bounds=PLOT_BOUNDS, show_density=False, title="")
track.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, title="")
ax.plot(manual_traj.x[0, :], manual_traj.x[1, :], "c--", linewidth=1.2, label="manual solve")
ax.legend(loc="upper left")
ax.set_title("Single-horizon plan (manual pipeline)")
plt.show()

## Step 10 — `TrajectoryOptimizationPlanner` wraps the same chain

`compute_solution()` is exactly steps 9a–9d orchestrated:

| Manual | Planner method |
| --- | --- |
| resolve initial guess | `_resolve_initial_guess()` |
| `transcription.transcribe()` | same |
| `pack_initial_guess()` | same |
| `Optimizer(...)` | `_make_optimizer()` |
| `optimizer.solve()` | same |
| `reconstruct_result()` | same |

The planner also caches `last_program`, `last_optimizer`, and `last_optimization_result` for inspection.

In [ ]:
planner = TrajectoryOptimizationPlanner(
    problem,
    transcription=transcription,
    options=trajopt_options,
)
planner_traj = planner.compute_solution(initial_guess=guess)
wrapper_result = planner.last_optimization_result

print("Planner wrapper vs manual pipeline:")
print(f"  n_z match: {planner.last_program.n_z == program.n_z}")
print(f"  J* manual  = {manual_result.cost:.6f}")
print(f"  J* planner = {wrapper_result.cost:.6f}")
print(f"  xy close: {np.allclose(planner_traj.x[0:2], manual_traj.x[0:2], atol=1e-3)}")

## Step 11 — Workspace cost heatmaps

`sample_field_costs` evaluates each grid cell as a **workspace point** with `point_probe` costs (heading-independent). Use these plots to **tune weights** before or after running MPC — they show what the optimizer "sees" in XY, not a guarantee of the executed path.

In [ ]:
path_cost_viz = track.distance_field(probe).as_cost(
    weight=PATH_COST_WEIGHT, shaping=quadratic_excess(threshold=0.1)
)
corridor_cost_viz = track.corridor_field(probe).as_cost(
    weight=CORRIDOR_COST_WEIGHT, shaping=quadratic_hinge(threshold=0.0)
)
obstacle_cost_viz = scene.clearance_field(probe).as_cost(
    weight=OBSTACLE_REPULSION_WEIGHT, shaping=inverse_barrier(epsilon=0.08)
)
cost_bounds = pad_bounds(PLOT_BOUNDS, COST_FIELD_MARGIN - PLOT_MARGIN)
_cost_kw = dict(bounds=cost_bounds, state_dim=sys_mpc.n, grid=(100, 100))
layers = {
    "path": sample_field_costs([path_cost_viz], **_cost_kw),
    "corridor": sample_field_costs([corridor_cost_viz], **_cost_kw),
    "obstacle": sample_field_costs([obstacle_cost_viz], **_cost_kw),
    "combined": sample_field_costs(
        [path_cost_viz, corridor_cost_viz, obstacle_cost_viz], **_cost_kw
    ),
}

_, ax = scene.plot(show=False, bounds=PLOT_BOUNDS, show_density=True, title="")
track.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, title="")
ax.set_title("Scene + reference track")
plt.show()

plot_cost_field_exports(
    layers,
    track=track,
    scene=scene,
    overlay_bounds=PLOT_BOUNDS,
    log_scale=True,
)

---

## MPC closed loop

Receding horizon: every `MPC_DT` seconds, measure `x`, solve a **new** `PlanningProblem` over `[t, t + MPC_HORIZON]`, apply only `u(:,0)`, integrate the plant until the next tick.

```
t=0        solve ──► apply u₀ ──► simulate ──► t=Δt
t=Δt       solve ──► apply u₀ ──► simulate ──► t=2Δt
           (warm-start guess from shifted previous plan)
```

Each tick uses `TrajectoryOptimizationPlanner` — the wrapper from Step 10.

In [ ]:
TF_SIM = 8.0
VX0 = 8.0
MPC_HZ = 5.0
SIM_HZ = 200.0
MPC_DT = 1.0 / MPC_HZ
SIM_DT = 1.0 / SIM_HZ
SUBSTEPS = max(1, int(round(MPC_DT / SIM_DT)))

sys_sim = JaxDynamicBicycleRateInputs()
sys_sim.params["mass"] = 1.03 * sys_mpc.params["mass"]
sys_sim.params["inertia"] = 1.02 * sys_mpc.params["inertia"]
for sys in (sys_mpc, sys_sim):
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = W_REAR_MAX
    sys.state.lower_bound[7] = -DELTA_MAX
    sys.state.upper_bound[7] = DELTA_MAX
    sys.inputs["w_rear_dot"].lower_bound[0] = -W_REAR_DOT_MAX
    sys.inputs["w_rear_dot"].upper_bound[0] = W_REAR_DOT_MAX
    sys.inputs["delta_dot"].lower_bound[0] = -DELTA_DOT_MAX
    sys.inputs["delta_dot"].upper_bound[0] = DELTA_DOT_MAX

s_start, _ = track.path.project(WAYPOINTS[0])
tangent = track.path.tangent(s_start)
theta0 = float(np.arctan2(tangent[1], tangent[0]))
x0 = np.array(
    [WAYPOINTS[0, 0], WAYPOINTS[0, 1], theta0, VX0, 0.0, 0.0, VX0 / r_r, 0.0]
)
sim_evaluator = sys_sim.compile(backend="jax", verbose=False)

In [ ]:
t_hist = [0.0]
x_hist = [x0.copy()]
u_hist = [np.zeros(sys_sim.m)]
mpc_plans = []
x = x0.copy()
t = 0.0
u_hold = np.zeros(sys_sim.m)
prev_plan = None
next_mpc_t = 0.0

while t < TF_SIM - 1e-12:
    if t >= next_mpc_t - 1e-12:
        tick_problem = PlanningProblem(sys=sys_mpc, x_start=x, X=X, cost=cost)
        tick_planner = TrajectoryOptimizationPlanner(
            tick_problem,
            transcription=transcription,
            options=trajopt_options,
        )
        guess = None
        if prev_plan is not None and prev_plan.n_samples >= 3:
            t_shift = prev_plan.t + MPC_DT
            mask = t_shift <= t + MPC_HORIZON + 1e-9
            if np.count_nonzero(mask) >= 3:
                x_guess = prev_plan.x[:, mask].copy()
                x_guess[:, 0] = x
                guess = Trajectory(
                    t=t_shift[mask] - t,
                    x=x_guess,
                    u=prev_plan.u[:, mask],
                )
        else:
            guess = default_initial_trajectory(
                tick_problem,
                transcription.initial_guess_time_grid(tick_problem),
            )
        plan = tick_planner.compute_solution(initial_guess=guess)
        res = tick_planner.last_optimization_result
        print(
            f"MPC @ t={t:.2f}s  success={res.success}  solve={res.solve_time_s:.3f}s"
        )
        prev_plan = plan
        u_hold = plan.u[:, 0].copy()
        mpc_plans.append(
            (t, Trajectory(t=plan.t + t, x=plan.x.copy(), u=plan.u.copy()))
        )
        next_mpc_t += MPC_DT

    for _ in range(SUBSTEPS):
        if t >= TF_SIM:
            break
        x = sim_evaluator.rk4_step(x, u_hold, t, SIM_DT)
        t += SIM_DT
        t_hist.append(t)
        x_hist.append(x.copy())
        u_hist.append(u_hold.copy())

traj = Trajectory(
    t=np.asarray(t_hist), x=np.asarray(x_hist).T, u=np.asarray(u_hist).T
)

## Results

Executed path on the scene, then state/input histories.

In [ ]:
_, ax = scene.plot(
    show=False,
    bounds=PLOT_BOUNDS,
    title="MPC on spatial scene",
    show_density=False,
)
track.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, title="")
ax.plot(traj.x[0, :], traj.x[1, :], color="tab:blue", linewidth=1.5, label="executed")
ax.legend(loc="upper left")
plt.show()

sys_sim.params = dict(sys_sim.params)
sys_sim.camera_scale = 14.0
sys_sim.traj = traj
sys_sim.plot_trajectory(signals=("x", "u"))

In [ ]:
history = SceneHistory(
    trail=TrajectoryPolyline(traj, window="prefix", color="b", style="--", linewidth=1.0),
    horizon=HorizonPolyline(mpc_plans, color="tab:orange", linewidth=2.0, style="--"),
)
sys_sim.animate(traj, overlays=[scene.as_visualizer(color="tab:red", opacity=0.45), history])

---

## Recap and extensions

**Pipeline in one line:** workspace geometry → state fields → `PlanningProblem` → transcribe → optimize → `Trajectory` → MPC loop.

| Next step | Where |
| --- | --- |
| Holonomic corridor trajopt (hard tube + goal) | `examples/scripts/trajectory_optimization/demo_holonomic_corridor.py` |
| Scene-only obstacle slalom MPC | `examples/scripts/mpc/demo_dynamic_bicycle_rate_mpc_multi_obstacle_scene.py` |
| Wide circuit + cost heatmaps | `examples/scripts/mpc/demo_dynamic_bicycle_rate_mpc_wide_circuit_lap.py` |
| Architecture contracts | `DESIGN.md` §6 (optimization/planning), spatial scene section |

**Try it (optional)**

1. Set `trajopt_options.solve_disp=True` and re-run Step 10 — full NLP preamble/report.
2. Remove hard `X` (keep soft costs only) and compare solve time / constraint violations.
3. Change one cost weight, re-run Step 11 heatmaps, then MPC — see how repulsion vs path tracking trades off.